In [1]:
# ---------------- Standard Libraries ----------------
import os
import pickle
from pathlib import Path
from datetime import datetime
import time
# ---------------- Data Handling ----------------
import numpy as np
import pandas as pd

# ---------------- Visualization ----------------
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({
    'font.size': 9,
    'font.weight': 'bold',
    'font.family': 'serif'
})

# ---------------- Machine Learning (sklearn) ----------------
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    roc_curve, auc, roc_auc_score, average_precision_score
)
# ---------------- Deep Learning (PyTorch) ----------------
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import _LRScheduler
from torchsummary import summary


# ---------------- File Handling ----------------
import h5py
import hdf5storage

# ---------------- Utilities ----------------
from tqdm import tqdm
import joblib
import shap

# ---------------- Custom Function ----------------
def load_pickle(filepath):
    """
    Load a Python object from a pickle file.

    Parameters
    ----------
    filepath : str
        Path to the .pkl file.

    Returns
    -------
    obj : any
        The Python object stored in the pickle file.
    """
    with open(filepath, 'rb') as f:
        obj = pickle.load(f)
    return obj

def extract_pid(filename):
    part = filename.split('_')[0]          
    pid_full = part.split('-')[1]          
    pid = pid_full[5:]                     
    return pid

def group_gcs(y):
    if y in [3,4,5,6,7,8]:
        return 2 # Bad Neurological state
    elif y in [9,10,11,12]:
        return 1 # Moderate Neurological state
    elif y in [13,14,15]:
        return 0 # Good Neurological state
    else:
        return None 

def group_rass(y):
    if y in [-5, -4]:
        return 2 # Bad Neurological state
    elif y in [-3, -2]:
        return 1 # Moderate Neurological state
    elif y in [-1, 0]:
        return 0 # Good Neurological state
    else:
        return None 
        
def group_icans(y):
    if y in [0]:
        return 0 # Good Neurological state
    elif y in [1, 2]:
        return 1 # Moderate Neurological state
    elif y in [3, 4]:
        return 2 # Bad Neurological state
    else:
        return None 

def group_cams(x):
    if 0 <= x <= 1:
        return 0 # Good Neurological state
    elif 2 <= x <= 5:
        return 1 # Moderate Neurological state
    else:
        return 2 # Bad Neurological state

In [2]:
# ------------------- CAMS - Dataset --------------------------
df_CAMS_metadata = pd.read_csv('/home/ayush/Desktop/CAM-S_dataset/CAMS_Training_Final_Metadata.csv')
df_CAMS_metadata["GroupedScore"] = df_CAMS_metadata["CAMS_SF"].apply(group_cams)
df_CAMS_metadata = df_CAMS_metadata.rename(columns={'CAMS_SF': 'RawScore'})
df_CAMS_metadata['Dataset'] = 'CAMS'
df_CAMS_metadata['TransformedScore'] = df_CAMS_metadata['RawScore']
df_CAMS_metadata = df_CAMS_metadata[['BDSPPatientID', 'Dataset', 'MorgothOutputFilename', 'RawScore', 'GroupedScore', 'TransformedScore']]

# ------------------- ICANS - Dataset --------------------------
df_ICANS_metadata = pd.read_csv('/home/ayush/Desktop/ICANS_dataset/ICANSTraining_Final_Metadata.csv')
df_ICANS_metadata["GroupedScore"] = df_ICANS_metadata["ICANS_raw"].apply(group_icans)
df_ICANS_metadata = df_ICANS_metadata.rename(columns={'StudyID':'BDSPPatientID',                                                      
                                                      'ICANS_raw': 'RawScore'})
df_ICANS_metadata['Dataset'] = 'ICANS'
df_ICANS_metadata['TransformedScore'] = df_ICANS_metadata['RawScore']
df_ICANS_metadata = df_ICANS_metadata[['BDSPPatientID', 'Dataset', 'MorgothOutputFilename', 'RawScore', 'GroupedScore', 'TransformedScore']]

# ------------------- GCS - Dataset --------------------------
df_GCS_metadata = pd.read_csv('/home/ayush/Desktop/MGB_GCS_EEGs/GCS_Filename_with_QC2.csv')
df_GCS_metadata['Filename'] = df_GCS_metadata['Filename'].astype(str) + '.csv'
df_GCS_metadata['GroupedScore'] = df_GCS_metadata['GCS_value'].apply(group_gcs)
df_GCS_metadata['BDSPPatientID'] = df_GCS_metadata['Filename'].apply(extract_pid)
df_GCS_metadata = df_GCS_metadata.rename(columns={'Filename': 'MorgothOutputFilename',
                                                  'GCS_value': 'RawScore'})
df_GCS_metadata['Dataset'] = 'GCS'
df_GCS_metadata['TransformedScore'] = 15-df_GCS_metadata['RawScore']
df_GCS_metadata = df_GCS_metadata[['BDSPPatientID', 'Dataset', 'MorgothOutputFilename', 'RawScore', 'GroupedScore', 'TransformedScore']]

# ------------------- RASS - Dataset --------------------------
df_RASS_metadata=pd.read_csv('/home/ayush/Desktop/MGB_RASS_EEGs/RASS_ALL_EEG_10min/RASS_files_updated_all.csv')
df_RASS_metadata['Filename'] = df_RASS_metadata['Filename'].str.rstrip("'") + '.csv'
df_RASS_metadata = df_RASS_metadata[
    ~df_RASS_metadata['RASS_value'].isin([1, 2, 3, 4])
]
df_RASS_metadata['GroupedScore'] = df_RASS_metadata['RASS_value'].apply(group_rass)
df_RASS_metadata['BDSPPatientID'] = df_RASS_metadata['Filename'].apply(extract_pid)
df_RASS_metadata['Dataset'] = 'RASS'
df_RASS_metadata = df_RASS_metadata.rename(columns={'Filename': 'MorgothOutputFilename',
                                                    'RASS_value': 'RawScore'})
df_RASS_metadata['TransformedScore'] = -df_RASS_metadata['RawScore']
df_RASS_metadata = df_RASS_metadata[['BDSPPatientID', 'Dataset', 'MorgothOutputFilename', 'RawScore', 'GroupedScore', 'TransformedScore']]

## ------------------ FULL DATASET (RASS- GCS - CAMS - ICANS) -------------------------
df_all_metadata = pd.concat(
    [df_RASS_metadata, df_GCS_metadata, df_CAMS_metadata, df_ICANS_metadata], # ],
    axis=0,
    ignore_index=True
)

df_icans_metadata = df_all_metadata[df_all_metadata['Dataset']=='ICANS']
df_icans_metadata

,BDSPPatientID,Dataset,MorgothOutputFilename,RawScore,GroupedScore,TransformedScore
208982,310,ICANS,310_seg2_SourceStart_2020-01-05 10:51:13.csv,2,1,2
208983,714,ICANS,714_seg41_SourceStart_2019-09-07 22:52:00.csv,3,2,3
208984,966,ICANS,966_seg8_SourceStart_2020-12-05 11:42:00.csv,3,2,3
208985,399,ICANS,399_seg3_SourceStart_2018-07-19 10:42:01.csv,2,1,2
208986,521,ICANS,521_seg0_SourceStart_2018-08-29 16:51:01.csv,2,1,2
...,...,...,...,...,...,...
209761,1670,ICANS,1670_seg4_SourceStart_2019-10-20 09:06:30.csv,0,0,0
209762,1215,ICANS,1215_seg5_SourceStart_2020-01-23 12:05:37.csv,0,0,0
209763,1962,ICANS,1962_seg1_SourceStart_2019-05-02 16:41:32.csv,0,0,0
209764,1916,ICANS,1916_seg0_SourceStart_2019-05-05 14:34:40.csv,0,0,0


In [3]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm


def morgoth_output_file_location(data_group):
    base_paths = {
        "RASS": "/home/ayush/Desktop/MGB_RASS_EEGs/RASS_ALL_EEG_10min/MORGOTH_embeddings",
        "GCS": "/home/ayush/Desktop/MGB_GCS_EEGs/MORGOTH_GCS_Embeddings",
        "CAMS": "/home/ayush/Desktop/CAM-S_dataset/MORGOTH_Embeddings",
        "ICANS": "/home/ayush/Desktop/ICANS_dataset/Morgoth_embeddings_for_model"
    }

    root = base_paths[data_group]

    return {
        "SLOWING": os.path.join(root, "SLOWING"),
        "FOCGEN": os.path.join(root, "FOCGEN"),
        "IIIC": os.path.join(root, "IIIC"),
        "NM": os.path.join(root, "NM"),
        "BS": os.path.join(root, "BS"),
        "SLEEP": os.path.join(root, "SLEEP")
    }


def load_feature(path, filename):
    df = pd.read_csv(os.path.join(path, filename))
    if "pred_class" in df.columns:
        df = df.drop(columns=["pred_class"])
    return df.values


def morgoth_10minfea_matrix(data_frame):
    features = []
    file_names = []

    for row in tqdm(data_frame.itertuples(index=False), total=len(data_frame)):
        fname = row.MorgothOutputFilename
        dataset = row.Dataset

        paths = morgoth_output_file_location(dataset)

        sleep = load_feature(paths["SLEEP"], fname)
        nm = load_feature(paths["NM"], fname)
        bs = load_feature(paths["BS"], fname)

        focgen = load_feature(paths["FOCGEN"], fname)
        slowing = load_feature(paths["SLOWING"], fname)
        iiic = load_feature(paths["IIIC"], fname)

        subject_feature = np.concatenate(
            [sleep, nm, bs, focgen, slowing, iiic],
            axis=1
        )

        features.append(subject_feature)
        file_names.append(fname)

    X = np.stack(features, axis=0)

    Y_grouped = data_frame["GroupedScore"].to_numpy()
    Y_raw = data_frame["RawScore"].to_numpy()
    Y_raw_transformed = data_frame['TransformedScore'].to_numpy()

    dataset_names = data_frame["Dataset"].to_numpy()
    file_names = np.array(file_names)

    print("Grouped Label shape:", Y_grouped.shape)
    print("Raw Label shape:", Y_raw.shape)
    print("Raw Label transformed shape:", Y_raw_transformed.shape)
    print("Feature matrix shape:", X.shape)
    
    return X, Y_grouped, Y_raw, Y_raw_transformed, dataset_names, file_names

In [4]:
X_data, Ygrouped, Yraw, Yraw_transformed, dataset_names,filenames = morgoth_10minfea_matrix(
    df_icans_metadata)

X_data_trch = torch.tensor(X_data, dtype=torch.float32)
Ygrouped_trch = torch.tensor(Ygrouped, dtype=torch.long)
Yraw_trch = torch.tensor(Yraw, dtype=torch.long)
dataset_names = np.array(dataset_names)

100%|████████████████████████████████████████| 784/784 [00:04<00:00, 157.85it/s]

Grouped Label shape: (784,)
Raw Label shape: (784,)
Raw Label transformed shape: (784,)
Feature matrix shape: (784, 591, 17)


In [5]:
#-------------------------- ResNet-GAP only model -------------------------------------
import torch.nn.functional as F
class ResidualBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(out_ch)

        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(out_ch)

        # If channels differ → use 1x1 conv for skip
        self.shortcut = nn.Sequential()
        if in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1),
                nn.BatchNorm1d(out_ch)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out = out + identity
        return F.relu(out)

class MORGOTH_ResNet1D_onlyGAP(nn.Module):
    def __init__(self, num_features, filters=None, use_logit=True):
        super().__init__()
        self.use_logit = use_logit
        

        if filters is None:
            filters = [64, 128, 128, 256, 256]

        # Initial conv
        self.conv0 = nn.Conv1d(num_features, filters[0], kernel_size=7, padding=3)
        self.bn0 = nn.BatchNorm1d(filters[0])
        self.pool0 = nn.MaxPool1d(kernel_size=2)

        # ResNet blocks
        blocks = []
        in_ch = filters[0]
        for out_ch in filters:
            blocks.append(ResidualBlock1D(in_ch, out_ch))
            blocks.append(nn.MaxPool1d(kernel_size=2))
            in_ch = out_ch
        self.resnet_layers = nn.Sequential(*blocks)

        # GAP but DON'T squeeze yet
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(256, 40)
    def forward(self, x):
        if self.use_logit:
            eps = 1e-6
            x = torch.log((x + eps) / (1 - x + eps))

        x = x.permute(0, 2, 1)

        x = self.pool0(F.relu(self.bn0(self.conv0(x))))
        x = self.resnet_layers(x)

        # GAP: (B,C,1)
        x = self.gap(x).squeeze(-1)
        x = self.dropout(x)
        x = F.relu(x)
        x = self.fc(x)
        x = F.normalize(x, p=2, dim=1)
        return x


# ------------- NESI score model ---------------
import torch
import torch.nn as nn
import torch.nn.functional as F


class EEGScoringModel(nn.Module):
    def __init__(self, input_dim=40, hidden_dim=30, dropout=0.3):
        super(EEGScoringModel, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, 1)  # scalar badness score
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

In [6]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load triplet encoder
triplet_model_trained = MORGOTH_ResNet1D_onlyGAP(num_features=17).to(device)
triplet_model_trained.load_state_dict(
    torch.load(
        "/home/ayush/Desktop/NESI_review_material/Leave_one_Cohort_Out_Model/ModelCheckpoints/HoldoutICANS/ResNetGAP_HoldoutICANS_BestModel.pth",
        map_location=device,
        weights_only=True
    )
)
triplet_model_trained.eval()

# Load NESI scoring model
nesi_score_checkpoint = torch.load(
    "/home/ayush/Desktop/NESI_review_material/Leave_one_Cohort_Out_Model/ModelCheckpoints/HoldoutICANS/NESI_ICANS_Holdout_Best_model.pth",
    map_location=device,
    weights_only=True
)

nesi_score_model_trained = EEGScoringModel().to(device)
nesi_score_model_trained.load_state_dict(nesi_score_checkpoint["model_state_dict"])
nesi_score_model_trained.eval()

# Extract NESI scores
X_data_trch = torch.as_tensor(X_data, dtype=torch.float32)
loader = DataLoader(TensorDataset(X_data_trch), batch_size=128, shuffle=False)

nesi_scores = []

with torch.inference_mode():
    for (x_batch,) in loader:
        x_batch = x_batch.to(device)

        embeddings = triplet_model_trained(x_batch)
        scores = nesi_score_model_trained(embeddings)

        nesi_scores.append(scores.reshape(-1).cpu().numpy())

nesi_scores = np.concatenate(nesi_scores)

print("Input shape:", tuple(X_data_trch.shape))
print("NESI scores shape:", nesi_scores.shape)

Input shape: (784, 591, 17)
NESI scores shape: (784,)
NESI scores: [-1.01509786e+00 -2.85754263e-01 -2.18946729e-02  6.09975815e-01
 -5.51564515e-01 -3.85393471e-01 -2.81063348e-01  2.68369224e-02
 -1.84575129e+00 -1.46004617e+00 -1.98599458e+00  2.69296914e-01
 -1.01716280e+00 -1.35197401e+00 -1.04729760e+00 -1.06028450e+00
  1.06295999e-02  4.03699309e-01 -7.02842891e-01  3.37868363e-01
  7.92159140e-02 -1.12477183e+00 -2.40753341e+00 -2.56768733e-01
  3.69216770e-01 -2.04446483e+00  3.33902687e-01 -1.49520910e+00
 -1.11767602e+00 -1.04928207e+00 -3.07038337e-01 -5.09550512e-01
 -7.50878006e-02 -4.11633104e-01 -1.91979551e+00 -1.79902995e+00
 -3.46807212e-01 -5.26541770e-01 -1.59314597e+00  4.52436775e-01
 -4.67825472e-01 -1.97509980e+00 -2.49567702e-01 -1.92426786e-01
  6.47671334e-03  1.06656075e+00 -4.68638033e-01  2.16675210e+00
 -1.37295437e+00 -1.51315975e+00 -1.63942480e+00  2.12638557e-01
 -9.73175824e-01 -1.56429425e-01 -2.26264596e+00 -4.03737456e-01
 -1.53418362e+00  3.963

In [7]:
df_icans_metadata = df_icans_metadata.copy()

assert len(df_icans_metadata) == len(nesi_scores), "Mismatch between metadata rows and NESI scores"

df_icans_metadata["NESI_ICANSHoldout"] = nesi_scores

df_icans_metadata

,BDSPPatientID,Dataset,MorgothOutputFilename,RawScore,GroupedScore,TransformedScore,NESI_ICANSHoldout
208982,310,ICANS,310_seg2_SourceStart_2020-01-05 10:51:13.csv,2,1,2,-1.015098
208983,714,ICANS,714_seg41_SourceStart_2019-09-07 22:52:00.csv,3,2,3,-0.285754
208984,966,ICANS,966_seg8_SourceStart_2020-12-05 11:42:00.csv,3,2,3,-0.021895
208985,399,ICANS,399_seg3_SourceStart_2018-07-19 10:42:01.csv,2,1,2,0.609976
208986,521,ICANS,521_seg0_SourceStart_2018-08-29 16:51:01.csv,2,1,2,-0.551565
...,...,...,...,...,...,...,...
209761,1670,ICANS,1670_seg4_SourceStart_2019-10-20 09:06:30.csv,0,0,0,-2.240694
209762,1215,ICANS,1215_seg5_SourceStart_2020-01-23 12:05:37.csv,0,0,0,-2.327958
209763,1962,ICANS,1962_seg1_SourceStart_2019-05-02 16:41:32.csv,0,0,0,-2.310222
209764,1916,ICANS,1916_seg0_SourceStart_2019-05-05 14:34:40.csv,0,0,0,-2.122946


In [8]:
df_icans_metadata.to_csv('/home/ayush/Desktop/NESI_review_material/Leave_one_Cohort_Out_Model/ICANS_Holdout_Result.csv',
                        index= False)